
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>



# Python User-Defined Functions

##### Objectives
1. Define a function
1. Create and apply a UDF
1. Create and register a UDF with Python decorator syntax
1. Create and apply a Pandas (vectorized) UDF

##### Methods
- <a href="https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.functions.udf.html" target="_blank">Python UDF Decorator</a>: **`@udf`**
- <a href="https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.functions.pandas_udf.html" target="_blank">Pandas UDF Decorator</a>: **`@pandas_udf`**

In [0]:
%run ./Includes/Classroom-Setup-02.7B

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


Resetting the learning environment:
| No action taken

Skipping install of existing datasets to "dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04"

Validating the locally installed datasets:
| listing local files...(7 seconds)
| validation completed...(7 seconds total)

Creating & using the schema "shifajamali55_soeb_da_dewd" in the catalog "hive_metastore"...(0 seconds)

Cloning the "sales" table from "dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/ecommerce/delta/sales_hist"....(6 seconds)

Predefined tables in "shifajamali55_soeb_da_dewd":
| sales

Predefined paths variables:
| DA.paths.working_dir: dbfs:/mnt/dbacademy-users/shifajamali55@gmail.com/data-engineering-with-databricks
| DA.paths.user_db:     dbfs:/mnt/dbacademy-users/shifajamali55@gmail.com/data-engineering-with-databricks/database.db
| DA.paths.datasets:    dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04

Setup completed (16 seconds)



### User-Defined Function (UDF)
A custom column transformation function

- Can’t be optimized by Catalyst Optimizer
- Function is serialized and sent to executors
- Row data is deserialized from Spark's native binary format to pass to the UDF, and the results are serialized back into Spark's native format
- For Python UDFs, additional interprocess communication overhead between the executor and a Python interpreter running on each worker node


For this demo, we're going to use the sales data.

In [0]:
sales_df = spark.table("sales")
display(sales_df)

order_id,email,transaction_timestamp,total_item_quantity,purchase_revenue_in_usd,unique_items,items
285712,wbrown@gonzales-miranda.com,1592521889512254,2,1071.0,1,"List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 1071.0, 595.0, 2))"
277921,campbellkatrina@phillips-duarte.com,1592458670364116,1,850.5,1,"List(List(NEWBED10, M_STAN_F, Standard Full Mattress, 850.5, 945.0, 1))"
274566,gregorytorres@meyer.com,1592419954844631,1,535.5,1,"List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1))"
257606,ryanolson@brooks.com,1592213705353885,2,1754.0,2,"List(List(null, M_PREM_F, Premium Full Mattress, 1695.0, 1695.0, 1), List(null, P_FOAM_S, Standard Foam Pillow, 59.0, 59.0, 1))"
257628,kimberly68@mcpherson.net,1592214238807084,1,1045.0,1,"List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))"
289539,levans13@hotmail.com,1592572270401950,2,2016.0,2,"List(List(NEWBED10, M_STAN_K, Standard King Mattress, 1075.5, 1195.0, 1), List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1))"
257794,anthonylopez@gmail.com,1592218840420069,1,1045.0,1,"List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))"
257918,wlarson@sanchez.info,1592221370370320,1,1695.0,1,"List(List(null, M_PREM_F, Premium Full Mattress, 1695.0, 1695.0, 1))"
257798,hernandezjohnny@ball.com,1592218929139881,1,1695.0,1,"List(List(null, M_PREM_F, Premium Full Mattress, 1695.0, 1695.0, 1))"
258127,laura40@reynolds.com,1592224798871113,1,1195.0,1,"List(List(null, M_STAN_K, Standard King Mattress, 1195.0, 1195.0, 1))"



### Define a function

Define a function (on the driver) to get the first letter of a string from the **`email`** field.

In [0]:
def first_letter_function(email):
    return email[0]

first_letter_function("annagray@kaufman.com")

'a'


### Create and apply UDF
Register the function as a UDF. This serializes the function and sends it to executors to be able to transform DataFrame records.

In [0]:
first_letter_udf = udf(first_letter_function)


Apply the UDF on the **`email`** column.

In [0]:
from pyspark.sql.functions import col

display(sales_df.select(first_letter_udf(col("email"))))

first_letter_function(email)
w
c
g
r
k
l
a
w
h
l



### Use Decorator Syntax (Python Only)

Alternatively, you can define and register a UDF using <a href="https://realpython.com/primer-on-python-decorators/" target="_blank">Python decorator syntax</a>. The **`@udf`** decorator parameter is the Column datatype the function returns.

You will no longer be able to call the local Python function (i.e., **`first_letter_udf("annagray@kaufman.com")`** will not work).

<img src="https://files.training.databricks.com/images/icon_note_32.png" alt="Note"> This example also uses <a href="https://docs.python.org/3/library/typing.html" target="_blank">Python type hints</a>, which were introduced in Python 3.5. Type hints are not required for this example, but instead serve as "documentation" to help developers use the function correctly. They are used in this example to emphasize that the UDF processes one record at a time, taking a single **`str`** argument and returning a **`str`** value.

In [0]:
# Our input/output is a string
@udf("string")
def first_letter_udf(email: str) -> str:
    return email[0]


And let's use our decorator UDF here.

In [0]:
from pyspark.sql.functions import col

sales_df = spark.table("sales")
display(sales_df.select(first_letter_udf(col("email"))))

first_letter_udf(email)
w
c
g
r
k
l
a
w
h
l



### Pandas/Vectorized UDFs

Pandas UDFs are available in Python to improve the efficiency of UDFs. Pandas UDFs utilize Apache Arrow to speed up computation.

* <a href="https://databricks.com/blog/2017/10/30/introducing-vectorized-udfs-for-pyspark.html" target="_blank">Blog post</a>
* <a href="https://spark.apache.org/docs/latest/api/python/user_guide/sql/arrow_pandas.html?highlight=arrow" target="_blank">Documentation</a>

<img src="https://databricks.com/wp-content/uploads/2017/10/image1-4.png" alt="Benchmark" width ="500" height="1500">

The user-defined functions are executed using: 
* <a href="https://arrow.apache.org/" target="_blank">Apache Arrow</a>, an in-memory columnar data format that is used in Spark to efficiently transfer data between JVM and Python processes with near-zero (de)serialization cost
* Pandas inside the function, to work with Pandas instances and APIs

<img src="https://files.training.databricks.com/images/icon_warn_32.png" alt="Warning"> As of Spark 3.0, you should **always** define your Pandas UDF using Python type hints.

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

# We have a string input/output
@pandas_udf("string")
def vectorized_udf(email: pd.Series) -> pd.Series:
    return email.str[0]

# Alternatively
# def vectorized_udf(email: pd.Series) -> pd.Series:
#     return email.str[0]
# vectorized_udf = pandas_udf(vectorized_udf, "string")

In [0]:
display(sales_df.select(vectorized_udf(col("email"))))

vectorized_udf(email)
w
c
g
r
k
l
a
w
h
l



We can register these Pandas UDFs to the SQL namespace.

In [0]:
spark.udf.register("sql_vectorized_udf", vectorized_udf)

<function __main__.vectorized_udf(email: pandas.core.series.Series) -> pandas.core.series.Series>

In [0]:
%sql
-- Use the Pandas UDF from SQL
SELECT sql_vectorized_udf(email) AS firstLetter FROM sales

firstLetter
w
c
g
r
k
l
a
w
h
l



### Clean up classroom

In [0]:
DA.cleanup()

Resetting the learning environment:
| dropping the schema "shifajamali55_soeb_da_dewd"...(1 seconds)
| removing the working directory "dbfs:/mnt/dbacademy-users/shifajamali55@gmail.com/data-engineering-with-databricks"...(0 seconds)

Validating the locally installed datasets:
| listing local files...(5 seconds)
| validation completed...(5 seconds total)



&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>